In [ ]:
import time
import os
import numpy as np
import cv2
from PIL import Image
import torch
import albumentations as A
from torch.utils.data import DataLoader
from torchvision.transforms import functional as F
from tqdm.auto import tqdm
from typing import List, Dict, Tuple, Union, Final

## Configurations
### Model parameters

In [ ]:
# number of classes (including background)
# the order of objects when creating the semnatic masks is important for semantic segmentation
# we create the semnatic masks in this order: bg, cage, cell, and then bead
# as cells can be inside cages (creating holes in cage masks), and beads
# can potentially be over the cells (creating holes)
# we include background (index 0) as it is needed in the model definition of Hugging Face
LABEL_MAP: Dict[int, str] = {0: 'bg', 1: 'cage', 2: 'cell', 3: 'bead'}

MODEL_PATH = 'checkpoints'
if not os.path.exists(MODEL_PATH):
    os.mkdir(MODEL_PATH)

# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 518

TRANSFORM_MEAN: Final[np.ndarray] = np.array([0.485, 0.456, 0.406])
TRANSFORM_STD: Final[np.ndarray] = np.array([0.229, 0.224, 0.225])

### Dataset parameters
Use the script in `Semantic Segmentation over Cage Crops.ipynb` to create the cropped images for cages. This step should be repeated for any additional dataset/cell type available. Point to the location of these cropped imaged below. 

In [ ]:
CROPPED_IMAGES_FOLDER = 'cage_crops_data'

## Data Model
### Dataset class

In [ ]:
from sem_seg_utils import SemanticMaskDataset

### Image and segmentation mask transforms
Here, we use albumentations package that takes both image and annotations to apply the transformations on them. It takes and returns numpy arrays. 

In [ ]:
train_transform = A.Compose([A.HorizontalFlip(), 
                             A.VerticalFlip(), 
                             A.GridDistortion(p=0.2), 
                             A.RandomBrightnessContrast(brightness_limit = (-0.2, 0.2), 
                                                        contrast_limit = (-0.2, 0.2)),
                             A.GaussNoise()])

### Datasets and Dataloaders

In [ ]:
# datasets
train_dataset = SemanticMaskDataset(images_path=os.path.join(CROPPED_IMAGES_FOLDER, "images", "train"), 
                                    masks_path=os.path.join(CROPPED_IMAGES_FOLDER, "masks", "train"),
                                    mean=TRANSFORM_MEAN, 
                                    std=TRANSFORM_STD,
                                    model_input_size=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE),
                                    transform=train_transform)

test_dataset = SemanticMaskDataset(images_path=os.path.join(CROPPED_IMAGES_FOLDER, "images", "test"), 
                                   masks_path=os.path.join(CROPPED_IMAGES_FOLDER, "masks", "test"),
                                   mean=TRANSFORM_MEAN, 
                                   std=TRANSFORM_STD,
                                   model_input_size=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)
                                  )

## Visualization

In [ ]:
from sem_seg_utils import show_sample

In [ ]:
img = show_sample(32, train_dataset)
Image.fromarray(img[:, :, ::-1])

## Model Definition

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

from transformers import Dinov2Config, Dinov2Model, Mask2FormerConfig, Mask2FormerForUniversalSegmentation

def get_mask2former_semantic_segmentation_model_with_dinov2_backbone(
    id2label: Dict[int, str], 
    model_input_size: int, 
    model_type: str, 
    with_registers: bool
):

    # transformer layer outputs to use
    output_indices_map: Dict[str, List[int]] = {
        "small": [6, 8, 10, 12], 
        "base":  [6, 8, 10, 12], 
        "large": [18, 20, 22, 24], 
        "giant": [34, 36, 38, 40]
    }
    
    if model_type.lower() in output_indices_map.keys():
        if with_registers:
            dinov2_checkpoint_str: str = "dinov2-with-registers-" + model_type.lower()
        else:
            dinov2_checkpoint_str: str = "dinov2-" + model_type.lower()
        
        output_indices: List[int] = output_indices_map[model_type.lower()] 
    else:
        dinov2_checkpoint_str: str = "dinov2-base"
        output_indices: List[int] = output_indices_map["base"]
        print(f"[ERROR] Incorrect model type passed {model_type}! Using the base model by default.")
        
        

    # store Dinov2 weights locally to reload them again, only do it if already not loaded locally
    if not os.path.exists(os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth")):
        dinov2_model = Dinov2Model.from_pretrained("facebook/" + dinov2_checkpoint_str, out_indices=output_indices)
        torch.save(dinov2_model.state_dict(), os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth"))

    # create Mask2Former config for semantic segmentation with Dinov2 backbone
    
    # mask2former_checkpoint = "facebook/mask2former-swin-large-ade-semantic"
    mask2former_checkpoint = "facebook/mask2former-swin-tiny-ade-semantic"
    
    model_config = Mask2FormerConfig.from_pretrained(mask2former_checkpoint)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_checkpoint,
                                                                id2label=id2label,
                                                                ignore_mismatched_sizes=True)
    model_config = model.config
    model_config.backbone_config = Dinov2Config.from_pretrained("facebook/" + dinov2_checkpoint_str, out_indices=output_indices)

    

    
    # instantiate Mask2Former model with Dinov2 backbone (random weights)
    model = Mask2FormerForUniversalSegmentation(model_config)

    # load Dinov2 weights into Mask2Former backbone
    dinov2_backbone = model.model.pixel_level_module.encoder
    dinov2_backbone.load_state_dict(torch.load(os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth")))

    # freeze all the weights in Dinov2 backbone
    # for param in dinov2_backbone.parameters():
    #     param.requires_grad_(False)

    # this is for freezing the backbone in Mask2Former, it should be the same as above
    for param in model.model.pixel_level_module.encoder.parameters():
        param.requires_grad_(False)

    return model

In [ ]:
model = get_mask2former_semantic_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP, 
    model_input_size=MODEL_INPUT_SIZE, 
    model_type="large", 
    with_registers=False
)

## Training
### Training parameters

In [ ]:
# training batch size
# this should be at least 2 as the DeepLab model Batch norm require at least 2
BATCH_SIZE: Final[int] = 16
# learning rate
LEARNING_RATE: Final[float] = 1e-5
# number of training epochs
NUM_EPOCHS = 2
# learning rate decay steps, a value of 0 means One-cycle LR scheduler should be used
LR_DECAY_STEPS = 2

### Dataloaders

In [ ]:
from transformers import Mask2FormerImageProcessor

# we need to convert the masks to a set of binary masks and a bunch of classes for training Mask2Fomer
# to simplify, we use the already implemented Higging Face preprocessor for this conversion
# we pass all the other flags as False as the image is already augmented and normalized

hg_preprocessor = Mask2FormerImageProcessor(ignore_index=-1, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)
     
def collate_fn(batch):
    inputs = list(zip(*batch))
    images = inputs[0]
    segmentation_masks = inputs[1]
    # this function pads the inputs to the same size,
    # and creates a pixel mask
    # actually padding isn't required here since we have already created image tensors of the same size
    # TODO: images and segmentation_masks are both tensors (returned by the Data class, hg_preprocessor
    # seems to be working fine with tensors, but we can return everything as numpy arrays from the class
    batch = hg_preprocessor(
        images,
        segmentation_maps=segmentation_masks,
        return_tensors="pt",
    )
    # return the semantic segmentation masks for evaluation as well
    batch['semantic_mask_tensor'] = torch.stack(segmentation_masks, dim=0)
    return batch

# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, collate_fn=collate_fn)

### Optimizer, LR scheduler and loss function

In [ ]:
# construct an optimizer
# consider using AdamW optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr = LEARNING_RATE)

print(f"Adam Optimizer is configured for {NUM_EPOCHS} epochs")

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)


### Performance metrics

In [ ]:
from sem_seg_utils import mIoU_masks, pixel_accuracy_masks

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          preprocessor,
          optimizer, 
          lr_scheduler,
          num_epochs,
          device,
         ):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_ious: List[float] = []
    train_accs: List[float] = []
    test_ious: List[float] = [] 
    test_accs: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        iou_score: float = 0
        accuracy: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            # forward pass 
            outputs = model(
                pixel_values=data["pixel_values"].to(device),
                mask_labels=[labels.to(device) for labels in data["mask_labels"]],
                class_labels=[labels.to(device) for labels in data["class_labels"]],
            )
           
            # evaluate metrics
            # we check the performance here on the "resized" input images and masks
            # predict segmentation maps, we are using the Hugging Face post processing function
            predicted_segmentation_masks = preprocessor.post_process_semantic_segmentation(
                outputs, 
                target_sizes=[(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)] * BATCH_SIZE
            )
            # convert from list
            predicted_segmentation_masks = torch.stack(predicted_segmentation_masks, dim=0).to(torch.float32)

            # ground truth segmentation masks
            gt_segmentation_masks = data['semantic_mask_tensor'].to(device)
            
            iou_score += mIoU_masks(
                predicted_segmentation_masks, 
                gt_segmentation_masks, 
                num_classes=len(LABEL_MAP), 
                ignore_index_zero=False
            )
            
            accuracy += pixel_accuracy_masks(predicted_segmentation_masks, gt_segmentation_masks)
            
            # backward
            loss = outputs.loss
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_iou_score: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):
                # forward
                outputs = model(pixel_values=data["pixel_values"].to(device))

                # evaluation metrics
                predicted_segmentation_masks = preprocessor.post_process_semantic_segmentation(
                    outputs, 
                    target_sizes=[(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)] * BATCH_SIZE
                )
                predicted_segmentation_masks = torch.stack(predicted_segmentation_masks, dim=0).to(torch.float32)
                
                # ground truth segmentation masks
                gt_segmentation_masks = data['semantic_mask_tensor'].to(device)
                
                test_iou_score +=  mIoU_masks(predicted_segmentation_masks, 
                                              gt_segmentation_masks, 
                                              num_classes=len(LABEL_MAP), 
                                              ignore_index_zero=False
                                              )
                test_accuracy += pixel_accuracy_masks(predicted_segmentation_masks, gt_segmentation_masks)
                # no testing loss available
                # loss = outputs.loss                          
                # test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= len(train_loader)
        iou_score /= len(train_loader)
        accuracy /= len(train_loader)
        
        test_running_loss /= len(test_loader)
        test_iou_score /= len(test_loader)
        test_accuracy /= len(test_loader)
                       
         # save the results
        train_losses.append(running_loss)
        train_ious.append(iou_score)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_ious.append(test_iou_score)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join(MODEL_PATH, 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train mean IoU: {:.3f} \n".format(iou_score),
              "Test mean IoU: {:.3f} \n".format(test_iou_score),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_mean_iou' :train_ious, 'test_mean_iou': test_ious,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history  = train(model, 
                 train_loader, 
                 test_loader, 
                 hg_preprocessor,
                 optimizer, 
                 lr_scheduler,
                 NUM_EPOCHS,
                 device
                )

### Saving the best/final model with some model configurations

In [ ]:
BEST_CHECKPOINT = 'checkpoint_5.pt'
model = get_mask2former_semantic_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP, 
    model_input_size=MODEL_INPUT_SIZE, 
    model_type="large", 
    with_registers=False
)

model.load_state_dict(torch.load(os.path.join(MODEL_PATH, BEST_CHECKPOINT)))
model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['input_size'] = MODEL_INPUT_SIZE
torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

## Testing

In [ ]:
LABEL_MAP: Dict[int, str] = {0: 'bg', 1: 'cage', 2: 'cell', 3: 'bead'}

MODEL_PATH = 'checkpoints'
CHECKPOINT_NAME = 'checkpoint_5.pt'
# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 518


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = get_mask2former_semantic_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP, 
    model_input_size=MODEL_INPUT_SIZE, 
    model_type="large", 
    with_registers=False
)
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, CHECKPOINT_NAME)))

In [ ]:
def evaluate(model, data_loader, preprocessor, device):
    # run the validation
    model.eval()
    model.to(device)
    
    test_iou_score: float = 0
    test_accuracy: float = 0
        
    # validation loop
    with torch.no_grad():
        for i, data in enumerate(tqdm(test_loader)):
            # forward
            outputs = model(pixel_values=data["pixel_values"].to(device))

            # evaluation metrics
            predicted_segmentation_masks = preprocessor.post_process_semantic_segmentation(
                outputs, 
                target_sizes=[(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)] * BATCH_SIZE
            )
            predicted_segmentation_masks = torch.stack(predicted_segmentation_masks, dim=0).to(torch.float32)
                
            # ground truth segmentation masks
            gt_segmentation_masks = data['semantic_mask_tensor'].to(device)
                
            test_iou_score +=  mIoU_masks(predicted_segmentation_masks, 
                                          gt_segmentation_masks, 
                                          num_classes=len(LABEL_MAP), 
                                          ignore_index_zero=False
                                         )
            test_accuracy += pixel_accuracy_masks(predicted_segmentation_masks, gt_segmentation_masks)
            # no testing loss available
            # loss = outputs.loss                          
            # test_running_loss += loss.item()
            
    test_iou_score /= len(test_loader)
    test_accuracy /= len(test_loader)
                   
    print("Test mean IoU: {:.3f} \n".format(test_iou_score),
          "Test Accuracy: {:.3f} \n".format(test_accuracy))

In [ ]:
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, '20250129_sets_1_2_3_6_to_41_dinov2_mask2former_0p2_bbox_0p2_b_c_adj_16_bs_6_epochs_1cl_lrs_2e-4.pt'))['model_state_dict'])
evaluate(model, test_loader, hg_preprocessor, device)

In [ ]:
# MC-38
# Test mean IoU: 0.910 
# Test Accuracy: 0.985 

# ALL
# Test mean IoU: 0.915
# Test Accuracy: 0.989 

In [ ]:
from transformers import Mask2FormerImageProcessor

def predict(model, input_image, device):
    
    model.eval()
    model.to(device)

    input_shape = input_image.shape
    if len(input_shape) < 3:
        image = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
        
    org_img_height, org_img_width = input_shape[:2]
    
    
    hg_preprocessor = Mask2FormerImageProcessor(ignore_index=-1, 
                                                do_resize=True,
                                                size=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE),
                                                size_divisor=14,
                                                reduce_labels=False, 
                                                do_rescale=True,
                                                image_mean=TRANSFORM_MEAN,
                                                image_std=TRANSFORM_STD,
                                                do_normalize=True)
       
    processed_img_dict = hg_preprocessor(image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_img_dict["pixel_values"].to(device))
        predicted_segmentation_masks = hg_preprocessor.post_process_semantic_segmentation(
                outputs, 
                target_sizes=[(org_img_height, org_img_width)]
            )
    return predicted_segmentation_masks[0].cpu().numpy().astype(np.uint8)

def predict_batch(model, input_images_list, device):
    
    model.eval()
    model.to(device)

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_list: List[np.array] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            images_list.append(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        else:
            images_list.append(img)
        org_img_dims.append(img_shape[:2])
    
    hg_preprocessor = Mask2FormerImageProcessor(ignore_index=-1, 
                                                do_resize=True,
                                                size=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE),
                                                size_divisor=14,
                                                reduce_labels=False, 
                                                do_rescale=True,
                                                image_mean=TRANSFORM_MEAN,
                                                image_std=TRANSFORM_STD,
                                                do_normalize=True)
       
    processed_imgs_dict = hg_preprocessor(images_list, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_imgs_dict["pixel_values"].to(device))
        predicted_segmentation_masks = hg_preprocessor.post_process_semantic_segmentation(
                outputs, 
                target_sizes=org_img_dims
            )
    masks_array: np.array = torch.stack(predicted_segmentation_masks, dim=0).cpu().numpy().astype(np.uint8)
    return [mask for mask in masks_array]

In [ ]:
idx = 21932
img_t, mask_t = test_dataset[idx]
image: np.ndarray = img_t.permute(1, 2, 0).squeeze().numpy()
mask_gt: np.ndarray = mask_t.numpy().astype(np.uint8)
# scale back and add the mean, scale to 0-255
image = ((image * train_dataset.std + train_dataset.mean) * 255).mean(axis=2).astype(np.uint8)
mask = predict(model, image, device)

In [ ]:
Image.fromarray(show_sample(idx, test_dataset))

In [ ]:
Image.fromarray(mask * 63)

In [ ]:
import time
start = time.time()
for i in range(100):
    mask = predict(model, image, device)
print(f"Mask2Former inference took {np.round((time.time() - start) * 10, 2)} ms")

In [ ]:
batch_size = 4
start = time.time()
for i in range(100):
    masks = predict_batch(model, [image] * batch_size, device)
print(f"Mask2Former inference took {np.round((time.time() - start) * 10, 2)} ms per batch of size {batch_size}")
print(f"Mask2Former inference took {np.round((time.time() - start) * 10 / batch_size, 2)} ms per image")